# Notebook 01 — The Memory Problem in LLM Agents
## A Survey with Experiments

**Series:** LLM Agent Memory — Week 1  
**Author:** Nemo  
**GitHub:** [llm-agent-memory-series](https://github.com/rohitashwachaks/llm-agent-memory-series)

---

## Why this notebook exists

Every serious LLM agent today has to answer one question: *what does the model know about what happened before?* The industry has converged on three rough answers — dump the history in, summarize it, or retrieve chunks of it — and each one fails in its own characteristic way.

This notebook does two things:

1. **Surveys** the memory landscape (MemGPT, HippoRAG, Memory³, A-MEM, Generative Agents) in one place, with one-line takeaways.
2. **Implements and benchmarks** the three dominant baselines on a controlled 20-turn task, measuring recall accuracy, token cost, and latency.

The point isn't to beat anyone yet. It's to establish a **shared vocabulary and measurement setup** for everything that follows in this series.

---

## TL;DR of findings (fill in after running)

- [ ] Naive history: accurate until token limit, then catastrophic
- [ ] Summarization: degrades smoothly but loses numeric/factual precision
- [ ] Vector RAG: excellent on some queries, brittle on others — write up when

*(Fill these in with real numbers once experiments are run. That's the LinkedIn hook.)*

## 1. The Memory Landscape

A compressed survey. One paragraph per system, ordered roughly by release.

### 1.1 Generative Agents (Park et al., 2023)
Stanford's simulation of 25 agents in a sandbox town. Introduced the idea of a **memory stream** with importance scoring and periodic reflection. Memories decay by recency, relevance, and importance — the first widely-cited three-factor retrieval score in LLM agents. [arxiv:2304.03442](https://arxiv.org/abs/2304.03442)

### 1.2 MemGPT (Packer et al., 2023)
Frames the context window as RAM and external storage as disk. The LLM itself issues function calls to page memories in and out. This is the OS-analogy paper — foundational for the "hierarchy of memory tiers" idea. [arxiv:2310.08560](https://arxiv.org/abs/2310.08560)

### 1.3 HippoRAG (Gutiérrez et al., 2024)
Biologically-inspired: mimics the hippocampal indexing theory. Uses a knowledge graph + Personalized PageRank over entity nodes to retrieve context for multi-hop questions. Interesting as an alternative to pure vector retrieval. [arxiv:2405.14831](https://arxiv.org/abs/2405.14831)

### 1.4 Memory³ (Yang et al., 2024)
Explicit memory as a first-class modality alongside implicit (weights) and working (context) memory. Argues that explicit memory is cheaper than fine-tuning and more scalable than context stuffing. [arxiv:2407.01178](https://arxiv.org/abs/2407.01178)

### 1.5 A-MEM (Xu et al., 2025)
Zettelkasten-inspired. Memories are notes that form links to each other; retrieval traverses the link graph. Closest published relative of what I want to build. [arxiv:2502.12110](https://arxiv.org/abs/2502.12110)

### 1.6 Survey: Zhang et al., 2024
*A Survey on the Memory Mechanism of Large Language Model based Agents.* Use this as the reference anchor for where the field is. [arxiv:2404.13501](https://arxiv.org/abs/2404.13501)

**What this series adds:** a *controlled, reproducible* head-to-head benchmark across these approaches, plus two novel angles — task-aware retrieval (NB 07) and aggressive decay (NB 08).

## 2. Setup

In [1]:
import os
import json
import time
from dataclasses import dataclass, field
from typing import Optional
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import anthropic
import tiktoken
from sentence_transformers import SentenceTransformer
import chromadb

load_dotenv()
sns.set_theme(style='whitegrid')

client = anthropic.Anthropic()
MODEL = 'claude-sonnet-4-5'  # or claude-opus-4-7 if budget allows
encoder = tiktoken.get_encoding('cl100k_base')  # close enough for counting

## 3. The Benchmark Task

We need something **simple, controllable, and diagnostic**. Real multi-turn benchmarks (LoCoMo, LongMemEval) are great but noisy for a first pass. Starting small.

**Design:** A 20-turn conversation where the user introduces **5 ground-truth facts** at scattered positions (turns 2, 5, 9, 13, 17), and at turn 20 asks questions that require recalling each one.

- Facts are factoids with specific values: dates, numbers, names — things summarization is known to mangle.
- Turns between facts are filler: on-topic chatter that pushes earlier facts out of the naive window when we later shrink it.
- We run the same script through each memory strategy and score: **recall accuracy** (did the final answer contain the right value?), **total tokens used**, **latency**.

In [ ]:
@dataclass
class Turn:
    role: str  # 'user' or 'assistant'
    content: str
    fact_id: Optional[str] = None  # tag if this turn introduces a ground-truth fact

@dataclass
class Fact:
    id: str
    statement: str   # how the user reveals it
    question: str    # how we test recall
    answer: str      # ground truth substring to match

FACTS = [
    Fact('fav_color', 'By the way, my favorite color is ultramarine.',
         'What is my favorite color?', 'ultramarine'),
    Fact('dog_name', 'Oh I meant to tell you — my dog is named Pascal.',
         "What is my dog's name?", 'Pascal'),
    Fact('birth_year', 'I was born in 1993, for context.',
         'What year was I born?', '1993'),
    Fact('allergy', 'I should mention I am allergic to cashews.',
         'What am I allergic to?', 'cashew'),
    Fact('city', 'I live in Arlington, Texas.',
         'What city do I live in?', 'Arlington'),
]

# TODO: generate 15 filler turns between the fact turns.
# Keep them generic ('what do you think about X', small talk) so they don't
# accidentally cue the facts.

In [ ]:
def build_conversation() -> list[Turn]:
    """Construct the 20-turn ground-truth conversation.
    Facts go at positions 2, 5, 9, 13, 17 (user turns).
    Fillers fill the gaps.
    """
    # TODO: implement. Return list[Turn] of length 20.
    raise NotImplementedError

def score_answer(response: str, fact: Fact) -> bool:
    """Simple substring match, case-insensitive. Good enough for v1.
    Upgrade to LLM-as-judge in Notebook 04."""
    return fact.answer.lower() in response.lower()

## 4. Memory Strategies

Three implementations, same interface. Each takes the conversation so far and a query, returns a prompt-ready context.

In [ ]:
class MemoryStrategy:
    name: str = 'base'
    def ingest(self, turn: Turn) -> None: ...
    def context_for_query(self, query: str) -> str: ...
    def token_count(self, query: str) -> int:
        return len(encoder.encode(self.context_for_query(query)))

In [ ]:
class NaiveHistory(MemoryStrategy):
    """Dump everything. Baseline of baselines."""
    name = 'naive'
    def __init__(self, token_budget: int = 8000):
        self.turns: list[Turn] = []
        self.token_budget = token_budget

    def ingest(self, turn):
        self.turns.append(turn)

    def context_for_query(self, query):
        # TODO: format as 'user: ... \n assistant: ...' and truncate to budget
        # keeping the most recent turns (FIFO eviction).
        raise NotImplementedError

In [ ]:
class RunningSummary(MemoryStrategy):
    """After every N turns, ask the LLM to compress history into a summary.
    Keep: running_summary + last_k_turns."""
    name = 'summary'
    def __init__(self, summarize_every: int = 4, keep_recent: int = 2):
        self.summary = ''
        self.buffer: list[Turn] = []
        self.summarize_every = summarize_every
        self.keep_recent = keep_recent

    def _summarize(self, old_summary: str, new_turns: list[Turn]) -> str:
        # TODO: call Claude with a careful summarization prompt.
        # Important: explicitly ask it to preserve specific values (names, numbers, dates).
        # This is where summarization quietly fails — document the prompt you use.
        raise NotImplementedError

    def ingest(self, turn):
        self.buffer.append(turn)
        if len(self.buffer) >= self.summarize_every + self.keep_recent:
            to_compress = self.buffer[:self.summarize_every]
            self.summary = self._summarize(self.summary, to_compress)
            self.buffer = self.buffer[self.summarize_every:]

    def context_for_query(self, query):
        # TODO: concatenate summary + recent buffer
        raise NotImplementedError

In [ ]:
class VectorRAG(MemoryStrategy):
    """Each turn is embedded and stored. At query time, retrieve top-k by cosine."""
    name = 'vector_rag'
    def __init__(self, top_k: int = 5, model: str = 'all-MiniLM-L6-v2'):
        self.embedder = SentenceTransformer(model)
        self.chroma = chromadb.Client().create_collection('nb01', get_or_create=True)
        self.top_k = top_k
        self._counter = 0

    def ingest(self, turn):
        # TODO: embed turn.content, upsert into chroma with metadata
        raise NotImplementedError

    def context_for_query(self, query):
        # TODO: embed query, retrieve top_k, format as context
        raise NotImplementedError

## 5. The Runner

Plays the ground-truth conversation through a strategy, then asks the 5 recall questions and scores.

In [ ]:
@dataclass
class RunResult:
    strategy: str
    fact_id: str
    correct: bool
    prompt_tokens: int
    latency_s: float
    response: str

def run_experiment(strategy: MemoryStrategy, conversation: list[Turn],
                   facts: list[Fact]) -> list[RunResult]:
    # 1. Feed conversation into strategy
    for turn in conversation:
        strategy.ingest(turn)
    # 2. For each fact, query and score
    results = []
    for fact in facts:
        context = strategy.context_for_query(fact.question)
        prompt = f"{context}\n\nUser: {fact.question}\nAssistant:"
        t0 = time.time()
        # TODO: call Claude with prompt, capture response
        response = '[PLACEHOLDER]'
        latency = time.time() - t0
        results.append(RunResult(
            strategy=strategy.name,
            fact_id=fact.id,
            correct=score_answer(response, fact),
            prompt_tokens=len(encoder.encode(prompt)),
            latency_s=latency,
            response=response,
        ))
    return results

## 6. Run All Strategies

In [ ]:
# TODO: run the experiment 3x per strategy (for variance) and collect into a DataFrame
# conversation = build_conversation()
# all_results = []
# for StrategyCls in [NaiveHistory, RunningSummary, VectorRAG]:
#     for trial in range(3):
#         strategy = StrategyCls()
#         all_results.extend(run_experiment(strategy, conversation, FACTS))
# df = pd.DataFrame([r.__dict__ for r in all_results])
# df.to_csv('../results/nb01_results.csv', index=False)
# df.head()

## 7. Analysis

Three plots to publish:
1. **Recall accuracy per strategy** (bar chart, facts-level)
2. **Token cost vs accuracy** (scatter — the efficient-frontier view)
3. **Which facts each strategy forgets** (heatmap — the diagnostic view, most interesting)

The heatmap is the hook for LinkedIn. If summarization forgets birth year but RAG forgets the allergy, that's a concrete, tweet-able insight.

In [ ]:
# TODO: plot 1 — recall accuracy per strategy
# TODO: plot 2 — token/accuracy scatter
# TODO: plot 3 — fact-level heatmap (rows: strategy, cols: fact_id, values: accuracy)
# Save all to ../results/figures/

## 8. Findings & Next Steps

*(Fill in after running. These are the LinkedIn bullets.)*

- **Finding 1:** ...
- **Finding 2:** ...
- **Finding 3:** ...

**What Notebook 02 will do:** zoom into the summarization failures specifically. If summary loses `1993` but keeps `Arlington`, *why*? Is it the token position? The prompt? The compression ratio? Isolating that failure mode is the next move.

---

## References

- Park et al., 2023. *Generative Agents: Interactive Simulacra of Human Behavior.* arxiv:2304.03442
- Packer et al., 2023. *MemGPT: Towards LLMs as Operating Systems.* arxiv:2310.08560
- Gutiérrez et al., 2024. *HippoRAG: Neurobiologically Inspired Long-Term Memory for LLMs.* arxiv:2405.14831
- Yang et al., 2024. *Memory³: Language Modeling with Explicit Memory.* arxiv:2407.01178
- Zhang et al., 2024. *A Survey on the Memory Mechanism of LLM based Agents.* arxiv:2404.13501
- Liu et al., 2023. *Lost in the Middle: How Language Models Use Long Contexts.* arxiv:2307.03172
- Lewis et al., 2020. *Retrieval-Augmented Generation for Knowledge-Intensive NLP.* arxiv:2005.11401